In [1]:
from pathlib import Path
import os
import pandas as pd
from nautilus_trader.persistence.catalog import ParquetDataCatalog
import re
from datetime import datetime
from pathlib import Path
import zipfile
import json
import gc

# For defining Instrument
import requests
import pandas as pd

from decimal import Decimal

from nautilus_trader.model.identifiers import InstrumentId, Symbol
from nautilus_trader.model.instruments import Instrument, CryptoPerpetual
from nautilus_trader.model.objects import Currency, Price, Quantity, Money
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.persistence.wranglers import OrderBookDeltaDataWrangler
from nautilus_trader.adapters.bybit.loaders import BybitOrderBookDeltaDataLoader

from nautilus_trader.adapters.bybit import BybitProductType


In [2]:
SYMBOL = "ETHUSDT"
EXCHANGE = "BYBIT"
INSTRUMENT_ID = f"{SYMBOL}-LINEAR.{EXCHANGE}"
DATA_DIR = Path(os.environ.get("DATA_DIR", "~/desktop/tmpMarketData/OrderBookData")).expanduser() / SYMBOL
CATALOG_DIR = Path(os.getcwd()).parent/"nautilusDataCatalog"

In [3]:
DATA_DIR

PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/OrderBookData/ETHUSDT')

In [4]:
import re
from pathlib import Path
from datetime import datetime, date

path = Path(DATA_DIR) 

# 2. Compile the regex pattern once
DATE_REGEX = re.compile(r"\d{4}-\d{2}-\d{2}")

def extract_file_date(file_path: Path) -> date:
    """Extracts the date component from the filename and returns a date object."""
    match = DATE_REGEX.search(file_path.name)
    if not match:
        raise ValueError(
            f"Pipeline Stop: Filename '{file_path.name}' does not contain a valid YYYY-MM-DD date stamp."
        )
    
    # Convert string match to actual date object safely
    return datetime.strptime(match.group(), "%Y-%m-%d").date()

# 3. Sort the files
raw_files = sorted(
    [f for f in path.iterdir() if f.is_file() and f.name.endswith(".data.zip")], 
    key=extract_file_date
)

# 4. Fixed assertion message to match the filter
assert raw_files, f"Unable to find any .data.zip files in directory {path}"

raw_files

FileNotFoundError: [Errno 2] No such file or directory: '/Users/damensavvasavvi/desktop/tmpMarketData/OrderBookData/ETHUSDT'

In [4]:
with zipfile.ZipFile(raw_files[0]) as z:
    inner_name = z.namelist()[0]

    with z.open(inner_name) as f:
        for _ in range(3):
            line = f.readline()
            record = json.loads(line)
            print(record)

{'topic': 'orderbook.200.ETHUSDT', 'type': 'snapshot', 'ts': 1775001601282, 'data': {'s': 'ETHUSDT', 'b': [['2104.08', '0.03'], ['2104.03', '0.01'], ['2104.00', '0.01'], ['2103.99', '0.04'], ['2103.98', '0.04'], ['2103.97', '5.09'], ['2103.95', '0.04'], ['2103.94', '0.05'], ['2103.93', '0.04'], ['2103.92', '0.02'], ['2103.90', '7.77'], ['2103.89', '0.02'], ['2103.88', '0.01'], ['2103.87', '0.01'], ['2103.86', '0.10'], ['2103.85', '16.32'], ['2103.84', '0.02'], ['2103.83', '0.26'], ['2103.82', '0.03'], ['2103.81', '2.38'], ['2103.80', '0.03'], ['2103.79', '0.03'], ['2103.78', '35.13'], ['2103.77', '0.02'], ['2103.76', '17.73'], ['2103.75', '15.91'], ['2103.74', '0.01'], ['2103.73', '0.24'], ['2103.72', '0.06'], ['2103.71', '4.51'], ['2103.70', '0.02'], ['2103.69', '0.01'], ['2103.68', '0.02'], ['2103.67', '0.02'], ['2103.66', '0.04'], ['2103.65', '36.36'], ['2103.64', '28.55'], ['2103.63', '8.87'], ['2103.62', '0.03'], ['2103.61', '0.01'], ['2103.60', '0.02'], ['2103.59', '8.82'], ['210

In [5]:
# Get instrument specs from Bybit API

def get_bybit_linear_instrument_info(symbol: str, testnet: bool = False) -> dict:
    base_url = "https://api-testnet.bybit.com" if testnet else "https://api.bybit.com"

    params = {
        "category": "linear",
        "symbol": symbol.upper(),
    }

    r = requests.get(
        f"{base_url}/v5/market/instruments-info",
        params=params,
        timeout=20,
    )
    r.raise_for_status()

    payload = r.json()

    if payload["retCode"] != 0:
        raise RuntimeError(payload)

    instruments = payload["result"]["list"]

    if not instruments:
        raise ValueError(f"No Bybit linear instrument found for {symbol}")

    return instruments[0]

info = get_bybit_linear_instrument_info(SYMBOL)

In [6]:
symbol = info["symbol"]              
base_coin = info["baseCoin"]         
quote_coin = info["quoteCoin"]      
settle_coin = info["settleCoin"]     

tick_size = info["priceFilter"]["tickSize"]
qty_step = info["lotSizeFilter"]["qtyStep"]

price_precision = int(info["priceScale"])
size_precision = abs(Decimal(qty_step).as_tuple().exponent)

min_qty = info["lotSizeFilter"]["minOrderQty"]
max_qty = info["lotSizeFilter"]["maxOrderQty"]
min_notional = info["lotSizeFilter"]["minNotionalValue"]

min_price = info["priceFilter"]["minPrice"]
max_price = info["priceFilter"]["maxPrice"]

max_leverage = Decimal(info["leverageFilter"]["maxLeverage"])
margin_init = Decimal("1") / max_leverage

In [7]:
CRYPTOPERP_INSTRUMENT = CryptoPerpetual(
    instrument_id=InstrumentId.from_str(f"{symbol}-LINEAR.BYBIT"),
    raw_symbol=Symbol(symbol),

    base_currency=Currency.from_str(base_coin),
    quote_currency=Currency.from_str(quote_coin),
    settlement_currency=Currency.from_str(settle_coin),

    is_inverse=False,

    price_precision=price_precision,
    size_precision=size_precision,

    price_increment=Price.from_str(tick_size),
    size_increment=Quantity.from_str(qty_step),

    multiplier=Quantity.from_str("1"),
    lot_size=Quantity.from_str("1"),

    min_quantity=Quantity.from_str(min_qty),
    max_quantity=Quantity.from_str(max_qty),

    min_notional=Money.from_str(f"{min_notional} {quote_coin}"),
    max_notional=None,

    min_price=Price.from_str(min_price),
    max_price=Price.from_str(max_price),

    margin_init=margin_init,
    margin_maint=Decimal("0"),

    maker_fee=Decimal("0.0002"),
    taker_fee=Decimal("0.00055"),

    ts_event=0,
    ts_init=0,

    info=info,
)

CRYPTOPERP_INSTRUMENT

CryptoPerpetual(id=ETHUSDT-LINEAR.BYBIT, raw_symbol=ETHUSDT, asset_class=CRYPTOCURRENCY, instrument_class=SWAP, quote_currency=USDT, is_inverse=False, price_precision=2, price_increment=0.01, size_precision=2, size_increment=0.01, multiplier=1, lot_size=1, margin_init=0.01, margin_maint=0, maker_fee=0.0002, taker_fee=0.00055, info={'symbol': 'ETHUSDT', 'contractType': 'LinearPerpetual', 'status': 'Trading', 'baseCoin': 'ETH', 'quoteCoin': 'USDT', 'launchTime': '1615766400000', 'deliveryTime': '0', 'deliveryFeeRate': '', 'priceScale': '2', 'leverageFilter': {'minLeverage': '1', 'maxLeverage': '100.00', 'leverageStep': '0.01'}, 'priceFilter': {'minPrice': '0.01', 'maxPrice': '199999.98', 'tickSize': '0.01'}, 'lotSizeFilter': {'maxOrderQty': '10000.00', 'minOrderQty': '0.01', 'qtyStep': '0.01', 'postOnlyMaxOrderQty': '10000.00', 'maxMktOrderQty': '2000.00', 'minNotionalValue': '5'}, 'unifiedMarginTrade': True, 'fundingInterval': 480, 'settleCoin': 'USDT', 'copyTrading': 'both', 'upperFund

In [35]:
def ingest_order_book(order_book_data_file, instrument: Instrument = CRYPTOPERP_INSTRUMENT):
    """Ingests a daily order book data file into the ParquetDataCatalog."""
    orderbook_df = BybitOrderBookDeltaDataLoader.load(order_book_data_file, product_type=BybitProductType.LINEAR)
    final_snapshot = orderbook_df[orderbook_df['action']=='CLEAR'].iloc[-1].name
    before_final_snapshot = orderbook_df[orderbook_df.index < final_snapshot]
    orderBookWrangler = OrderBookDeltaDataWrangler(instrument=instrument)
    deltas = orderBookWrangler.process(before_final_snapshot)
    deltas.sort(key=lambda x: x.ts_init)
    return deltas


In [36]:
catalog = ParquetDataCatalog(str(CATALOG_DIR))
catalog.write_data([CRYPTOPERP_INSTRUMENT])

for orderBookDataFile in raw_files:
    catalog.write_data(ingest_order_book(orderBookDataFile, CRYPTOPERP_INSTRUMENT)) #NOTE: this was originally run with skip disjoint check = True
    print(f"Successfully cataloged {orderBookDataFile}")

File /Users/damensavvasavvi/Desktop/NautilusTrader/project/nautilusDataCatalog/data/crypto_perpetual/ETHUSDT-LINEAR.BYBIT/1970-01-01T00-00-00-000000000Z_1970-01-01T00-00-00-000000000Z.parquet already exists, skipping write
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/OrderBookData/ETHUSDT/2026-04-01_ETHUSDT_ob200.data.zip
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/OrderBookData/ETHUSDT/2026-04-02_ETHUSDT_ob200.data.zip
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/OrderBookData/ETHUSDT/2026-04-03_ETHUSDT_ob200.data.zip
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/OrderBookData/ETHUSDT/2026-04-04_ETHUSDT_ob200.data.zip
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/OrderBookData/ETHUSDT/2026-04-05_ETHUSDT_ob200.data.zip
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/OrderBookData/ETHUSDT/2026-04-06_ETHUSDT_ob200.data.zip
Successfully cataloged /Users/dam

In [5]:
### Check time gap between files ###
import pyarrow.parquet as pq
import pyarrow.compute as pc
from pathlib import Path

def check_file_transitions(
    catalog_dir: Path | str, 
    data_type: str, 
    instrument_id: str,
    time_column: str = "ts_event"
):
    """
    Checks only the gaps BETWEEN separate Parquet files, skipping internal rows.
    """
    target_dir = Path(catalog_dir) / "data" / data_type / instrument_id
    
    if not target_dir.exists():
        raise FileNotFoundError(f"Catalog directory not found: {target_dir}")

    parquet_files = list(target_dir.glob("*.parquet"))
    if not parquet_files:
        print("No files found.")
        return

    # 1. Get the Start and End bounds for every file
    file_bounds = []
    for fp in parquet_files:
        try:
            # Read only the time column
            table = pq.read_table(fp, columns=[time_column])
            if table.num_rows == 0: continue
            
            # Grab exact start and end of the file
            ts_array = table[time_column].to_numpy()
            file_bounds.append({
                "name": fp.name,
                "start": ts_array[0],
                "end": ts_array[-1]
            })
        except Exception as e:
            print(f"Error reading {fp.name}: {e}")

    # 2. Sort them chronologically by their start time
    file_bounds.sort(key=lambda x: x["start"])

    # 3. Calculate the jumps between files
    print(f"\nScanning transitions across {len(file_bounds)} files...\n")
    print("-" * 75)
    print(f"{'File Transition':<45} | {'Gap (Seconds)':<15}")
    print("-" * 75)
    
    max_gap = 0
    
    for i in range(1, len(file_bounds)):
        prev_file = file_bounds[i - 1]
        curr_file = file_bounds[i]
        
        # Calculate gap from the end of the previous file to the start of the current one
        gap_ns = curr_file["start"] - prev_file["end"]
        gap_sec = gap_ns / 1_000_000_000  # Convert nanoseconds to seconds
        
        if gap_sec > max_gap:
            max_gap = gap_sec
            
        if gap_sec < 0:
            print(f"OVERLAP DETECTED: {prev_file['name']} -> {curr_file['name']} ({gap_sec}s)")
        else:
            # Log the transition. Change gap_sec > 0 to a higher number if you only want to see large gaps
            print(f"File {i} -> File {i+1} {'':<27} | {gap_sec:.3f} s")

    print("-" * 75)
    print(f"Largest transition gap: {max_gap:.3f} seconds\n")


# ==========================================
# Execution 
# ==========================================
if __name__ == "__main__":
    CATALOG_DIR = Path.cwd().parent / "nautilusDataCatalog"
    
    check_file_transitions(
        catalog_dir=CATALOG_DIR,
        data_type="order_book_deltas",  
        instrument_id="ETHUSDT-LINEAR.BYBIT"
    )

KeyboardInterrupt: 